In [1]:
import numpy as np
import time
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.12.2)
Hello from the pygame community. https://www.pygame.org/contribute.html


* This will be on dance beat timing - therefore in 8 beats instead of 4_4 timing from typical music counts
  

In [2]:
def get_musical_bpm(bpm_times):
    if len(bpm_times) < 2:
        return None, None
    bpm_deltas = [bpm_times[i] - bpm_times[i-1] for i in range(1,len(bpm_times))]
    musical_bpm = 1 / np.mean(bpm_deltas) * 60
    musical_bpm = np.round(musical_bpm)

    standard_error = np.std([1/bpm for bpm in bpm_deltas]*60) / np.sqrt(len(bpm_deltas))
    return musical_bpm , np.round(standard_error,4)

In [3]:
def get_resized_font(text, max_width):
    # Start with a default font size
    font_size = 36
    font = pygame.font.Font(None, font_size)
    
    # Keep adjusting the font size until the text fits within the max width
    text_surface = font.render(text, True, text_color)
    while text_surface.get_width() > max_width and font_size > 12:  # Stop at a minimum font size
        font_size -= 2  # Decrease the font size
        font = pygame.font.Font(None, font_size)
        text_surface = font.render(text, True, text_color)
    
    return font, text_surface

In [ ]:

# Initialize pygame
pygame.init()
screen = pygame.display.set_mode((600, 300))
pygame.display.set_caption("⏱️ Spacebar Tap Timer")

# Fonts
font = pygame.font.Font(None, 36)

# Tap storage
hit_times = []
bpm_times = []

# Colors
bg_color = (20, 20, 20)
box_color = (50, 50, 200)
box_hover = (70, 70, 255)
box_disabled = (30, 30, 80)
text_color = (255, 255, 255)

# Box setup
enter_box = pygame.Rect(50, 150, 200, 80)
space_box = pygame.Rect(350, 150, 200, 80)
reset_box = pygame.Rect(65, 250, 150, 50)
reset_hit_box = pygame.Rect(335, 250, 150, 50)

bpm_value = None

running = True
while running:
    screen.fill(bg_color)

    mouse_pos = pygame.mouse.get_pos()
    mouse_click = False
    space_pressed = False

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            mouse_click = True
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_q:
                running = False
            if event.key == pygame.K_SPACE:
                space_pressed = True

    # Handle SPACE box
    if len(bpm_times) >= 2:
        is_hovering = space_box.collidepoint(mouse_pos)
        if is_hovering:
            pygame.draw.rect(screen, box_hover, space_box)
        else:
            pygame.draw.rect(screen, box_color, space_box)
    
        if (mouse_click and is_hovering) or space_pressed:
            t = time.time()
            hit_times.append(t)
            print(f"✅ Space-tap (box or spacebar) at {t:.4f}")
    else:
        pygame.draw.rect(screen, box_disabled, space_box)

    # Handle BPM box
    if enter_box.collidepoint(mouse_pos):
        pygame.draw.rect(screen, box_hover, enter_box)
        if mouse_click:
            t = time.time()
            bpm_times.append(t)
            print(f"✅ BPM-tap (box) at {t:.4f}")
    else:
        pygame.draw.rect(screen, box_color, enter_box)

    # Handle RESET box
    if reset_box.collidepoint(mouse_pos):
        pygame.draw.rect(screen, box_hover, reset_box)
        if mouse_click:
            hit_times.clear()
            bpm_times.clear()
            print("🔁 Data reset!")
    else:
        pygame.draw.rect(screen, box_color, reset_box)

    if reset_hit_box.collidepoint(mouse_pos):
        pygame.draw.rect(screen, box_hover, reset_hit_box)
        if mouse_click:
            hit_times.clear()
            print("🔁 Data reset!")
    else:
        pygame.draw.rect(screen, box_color, reset_hit_box)

    
    # Text inside boxes
    space_text = font.render("Tap/ SpaceBar", True, text_color)
    screen.blit(space_text, (space_box.x + 10, space_box.y + 25))

    enter_text = font.render("BPM", True, text_color)
    screen.blit(enter_text, (enter_box.x + 70, enter_box.y + 25))

    reset_text = font.render("Reset", True, text_color)
    screen.blit(reset_text, (reset_box.x + 40, reset_box.y + 10))
    
    reset_text = font.render("Reset Hits", True, text_color)
    screen.blit(reset_text, (reset_hit_box.x + 10, reset_box.y + 10))

    
    # Calculate Values
    bpm_value, SE_bpm = get_musical_bpm(bpm_times)
    beats, error = HitTimer(np.array(hit_times), bpm_value).get_hit_sequence()

    # Show stats
    hit_count_text = font.render(f"Hit Counter: {len(hit_times)}", True, text_color)
    bpm_count_text = font.render("BPM: {} +/- {}".format(bpm_value,SE_bpm), True, text_color)
    quit_text = font.render("Press Q to quit", True, (150, 150, 150))
    font_for_sequence, hit_sequence_text = get_resized_font(f"Hit Sequence: {beats}", screen.get_width() - 40)

    screen.blit(bpm_count_text, (20, 20))
    screen.blit(hit_count_text, (20, 50))
    screen.blit(hit_sequence_text, (20, 80))
    screen.blit(quit_text, (20, 110))

    pygame.display.flip()

# Clean exit
pygame.quit()

In [18]:
beats_ = HitTimer(np.array(hit_times),musical_bpm).get_hit_sequence()

NameError: name 'musical_bpm' is not defined

In [112]:
# HT = HitTimer(time_deltas,musical_bpm).get_hit_sequence()
# beats,_ = HT.get_hit_sequence()

In [31]:
# musical_bpm = 30

# time_deltas_son = np.array([0,1,1,2,1.5,1.5,2,1,2,1.5,1.75])
# time_deltas_error_example = np.array([0,1.1,0.99,1.95,1.55,1.5,2,1,2,1.5,1.75])
# time_deltas_rumba = np.array([0,1,1,2,1.5,2,1.5,1,2,1.5,2])
# time_deltas

In [26]:
class HitTimer:
    def __init__(self,hit_times,musical_bpm,lowest_time_unit=0.5):
        time_deltas = [0] + [hit_times[i] - hit_times[i-1] for i in range(1,len(hit_times))]

        self.time_deltas = time_deltas
        self.musical_bpm = musical_bpm
        self.lowest_time_unit = lowest_time_unit
        
        #Error margin lowest_time_unit / 2

    def get_time_between_beats(self): 
         # Get song features
        dance_bpm = self.musical_bpm *2
        
        # Beats per second
        beats_per_second = dance_bpm / 60
        
        # Time between each dance beat
        self.time_between_beats = 1/ beats_per_second

    def get_hit_sequence(self,verbose = False):
        if len(self.time_deltas) < 2:
            return None,None

        self.get_time_between_beats()
        
        # Est. beat location 
        est_beat = self.time_deltas/self.time_between_beats
        est_beat_rounded = np.round(est_beat/self.lowest_time_unit) * self.lowest_time_unit 
        raw_counts = np.cumsum(est_beat_rounded)
        
        # Reduce counts back to 123-567 timing
        reduce = lambda x: np.remainder(x,8)
        self.normalised_counts = np.array([reduce(x) for x in raw_counts]) + 1
    
        # Error in timing
        error = est_beat - est_beat_rounded
    
        if verbose: 
            output_text = """
    Hit / Beat Timer ({} bpm)
    ----------------
    * Beat Sequence Breakdown : {},
    * Error in actual beat & user beat = {}
            """.format(self.musical_bpm,normalised_counts,np.round(error,2))
            print(output_text)

        output_hits = self.get_last_x_hits()
        return output_hits, error

    def get_last_x_hits(self, x = 20):
        output_hits = self.normalised_counts
        if len(self.normalised_counts) >x:
            output_hits = self.normalised_counts[-x:]
        return output_hits
    
    # TODO -- Replace time deltas with the rounded value to remove error compounding
    def get_hit_sequence_lenient(self,verbose = False):
        if len(self.time_deltas) < 2:
            return None,None

        self.get_time_between_beats()
        
        # Est. beat location 
        est_beat = self.time_deltas/self.time_between_beats
        est_beat_rounded = np.round(est_beat/self.lowest_time_unit) * self.lowest_time_unit 
        raw_counts = np.cumsum(est_beat_rounded)
        
        # Reduce counts back to 123-567 timing
        reduce = lambda x: np.remainder(x,8)
        normalised_counts = np.array([reduce(x) for x in raw_counts]) + 1
    
        # Error in timing
        error = est_beat - est_beat_rounded
    
        if verbose: 
            output_text = """
    Hit / Beat Timer ({} bpm)
    ----------------
    * Beat Sequence Breakdown : {},
    * Error in actual beat & user beat = {}
            """.format(self.musical_bpm,normalised_counts,np.round(error,2))
            print(output_text)
            
        return normalised_counts, error
    
    
    def match_known_beat(self):
        pass